# Surrogate Workflow for 4-Factor Tip-Gap Study with `pymoo`

This notebook reproduces the surrogate modeling workflow for the 4-factor axis short-edge study and uses `pymoo` for NSGA-II.

Inputs:
- `magnet_thickness_mm`
- `magnet_length_mm`
- `v_angle_deg`
- `tip_gap_mm`

Targets:
- `loaded_torque_mean`
- `loaded_torque_ripple_pct`
- `cogging_torque_pkpk`
- `loaded_force_max`

Workflow:
1. Load FEMM summary CSV
2. Train Random Forest surrogate
3. Check 5-fold CV R²
4. Predict 1000 LHS candidates and inspect Pareto behavior
5. Compute magnet usage
6. Run `pymoo` NSGA-II on surrogate (`maximize torque`, `minimize magnet area`)


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.multioutput import MultiOutputRegressor

from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize

BASE_DIR = Path('/Users/dohyunkim/Documents/Window')
SUMMARY_CSV = BASE_DIR / 'axis_shortedge_ccf_4factor_50point_tipgap_summary.csv'
OUT_DIR = BASE_DIR / 'notebook_outputs_surrogate_4factor_tipgap_pymoo'
OUT_DIR.mkdir(exist_ok=True)

FEATURES = ['magnet_thickness_mm', 'magnet_length_mm', 'v_angle_deg', 'tip_gap_mm']
TARGETS = ['loaded_torque_mean', 'loaded_torque_ripple_pct', 'cogging_torque_pkpk', 'loaded_force_max']

plt.style.use('default')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)


In [ ]:
df = pd.read_csv(SUMMARY_CSV)
df['completed_bool'] = df['completed_design'].astype(str).str.lower().isin(['true', '1'])
df = df.loc[df['completed_bool']].copy()
print('rows =', len(df))
df.head()

## Basic data check

In [ ]:
df[FEATURES + TARGETS].describe().T

## Train surrogate and check 5-fold CV

In [ ]:
x = df[FEATURES]
y = df[TARGETS]

model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=2,
        random_state=20260728,
    )
)

cv = KFold(n_splits=5, shuffle=True, random_state=20260728)
y_cv = cross_val_predict(model, x, y, cv=cv)

cv_r2 = pd.DataFrame({
    'target': TARGETS,
    'cv_r2': [r2_score(y.iloc[:, i], y_cv[:, i]) for i in range(len(TARGETS))],
})
cv_r2

In [ ]:
model.fit(x, y)

## Helper functions

In [ ]:
def latin_hypercube(bounds, n_samples, seed=20260728):
    rng = np.random.default_rng(seed)
    data = {}
    for name, (lo, hi) in bounds.items():
        bins = (np.arange(n_samples) + rng.random(n_samples)) / n_samples
        rng.shuffle(bins)
        data[name] = lo + bins * (hi - lo)
    return pd.DataFrame(data)


def pareto_mask_max_torque_min_ripple(torque, ripple):
    n = len(torque)
    keep = np.ones(n, dtype=bool)
    for i in range(n):
        dominated = (
            (torque >= torque[i])
            & (ripple <= ripple[i])
            & ((torque > torque[i]) | (ripple < ripple[i]))
        )
        dominated[i] = False
        if dominated.any():
            keep[i] = False
    return keep


def magnet_usage_area(length_mm, thickness_mm, v_angle_deg):
    piece = length_mm * thickness_mm * np.sin(np.deg2rad(v_angle_deg))
    return 8.0 * piece


## Predict 1000 LHS candidates

In [ ]:
bounds = {name: (float(df[name].min()), float(df[name].max())) for name in FEATURES}
candidates = latin_hypercube(bounds, n_samples=1000, seed=20260728)
y_pred = model.predict(candidates)

pred_df = candidates.copy()
for idx, target in enumerate(TARGETS):
    pred_df[target] = y_pred[:, idx]

pred_df['pareto_torque_ripple'] = pareto_mask_max_torque_min_ripple(
    pred_df['loaded_torque_mean'].to_numpy(),
    pred_df['loaded_torque_ripple_pct'].to_numpy(),
)
pred_df['total_magnet_area_mm2'] = magnet_usage_area(
    pred_df['magnet_length_mm'].to_numpy(),
    pred_df['magnet_thickness_mm'].to_numpy(),
    pred_df['v_angle_deg'].to_numpy(),
)

pareto_1000 = pred_df.loc[pred_df['pareto_torque_ripple']].sort_values(
    ['loaded_torque_mean', 'loaded_torque_ripple_pct'],
    ascending=[False, True],
)
print('Pareto count =', len(pareto_1000))
pareto_1000.head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(pred_df['loaded_torque_mean'], pred_df['loaded_torque_ripple_pct'], s=12, alpha=0.25, label='Predicted 1000')
ax.scatter(pareto_1000['loaded_torque_mean'], pareto_1000['loaded_torque_ripple_pct'], s=40, label='Predicted Pareto')
ax.scatter(df['loaded_torque_mean'], df['loaded_torque_ripple_pct'], s=36, alpha=0.9, label='Measured FEMM')
ax.set_xlabel('Average Torque [N.m]')
ax.set_ylabel('Torque Ripple [%]')
ax.set_title('Surrogate 1000-candidate Pareto: Torque vs Ripple')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## Sensitivity table (Spearman)

In [ ]:
sens = df[
    FEATURES + ['loaded_torque_mean', 'loaded_torque_ripple_pct', 'cogging_torque_pkpk', 'loaded_force_max']
].corr(method='spearman')

sens.loc[
    FEATURES,
    ['loaded_torque_mean', 'loaded_torque_ripple_pct', 'cogging_torque_pkpk', 'loaded_force_max'],
].round(3)

## `pymoo` NSGA-II on surrogate: maximize torque, minimize magnet area

In [ ]:
class TorqueMagnetAreaProblem(Problem):
    def __init__(self, model, xl, xu):
        super().__init__(n_var=4, n_obj=2, n_constr=0, xl=xl, xu=xu)
        self.model = model

    def _evaluate(self, X, out, *args, **kwargs):
        X_df = pd.DataFrame(X, columns=FEATURES)
        pred = self.model.predict(X_df)
        torque = pred[:, 0]
        area = magnet_usage_area(X[:, 1], X[:, 0], X[:, 2])
        # pymoo minimizes by default
        out['F'] = np.column_stack([-torque, area])
        out['pred'] = pred

xl = df[FEATURES].min().to_numpy(dtype=float)
xu = df[FEATURES].max().to_numpy(dtype=float)
problem = TorqueMagnetAreaProblem(model, xl=xl, xu=xu)


In [ ]:
algorithm = NSGA2(pop_size=300)
result = minimize(
    problem,
    algorithm,
    ('n_gen', 60),
    seed=20260728,
    verbose=False,
)

X = result.X
F = result.F
X_df = pd.DataFrame(X, columns=FEATURES)
pred = model.predict(X_df)

nsga_df = pd.DataFrame(X, columns=FEATURES)
for i, target in enumerate(TARGETS):
    nsga_df[target] = pred[:, i]
nsga_df['total_magnet_area_mm2'] = magnet_usage_area(
    nsga_df['magnet_length_mm'].to_numpy(),
    nsga_df['magnet_thickness_mm'].to_numpy(),
    nsga_df['v_angle_deg'].to_numpy(),
)
nsga_df = nsga_df.sort_values(['loaded_torque_mean', 'total_magnet_area_mm2'], ascending=[False, True]).reset_index(drop=True)

print('NSGA result rows =', len(nsga_df))
nsga_df.head(15)

In [ ]:
measured = df.copy()
measured['total_magnet_area_mm2'] = magnet_usage_area(
    measured['magnet_length_mm'].to_numpy(),
    measured['magnet_thickness_mm'].to_numpy(),
    measured['v_angle_deg'].to_numpy(),
)

torque = measured['loaded_torque_mean'].to_numpy()
area = measured['total_magnet_area_mm2'].to_numpy()
measured_keep = np.ones(len(measured), dtype=bool)
for i in range(len(measured)):
    dominated = (
        (torque >= torque[i])
        & (area <= area[i])
        & ((torque > torque[i]) | (area < area[i]))
    )
    dominated[i] = False
    if dominated.any():
        measured_keep[i] = False

measured_pareto = measured.loc[measured_keep].sort_values(
    ['loaded_torque_mean', 'total_magnet_area_mm2'],
    ascending=[False, True],
).reset_index(drop=True)

print('Measured torque-area Pareto count =', len(measured_pareto))
display(measured_pareto[['run', 'magnet_thickness_mm', 'magnet_length_mm', 'v_angle_deg', 'tip_gap_mm', 'loaded_torque_mean', 'total_magnet_area_mm2']])

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(measured['loaded_torque_mean'], measured['total_magnet_area_mm2'], s=42, alpha=0.8, label='Measured FEMM')
ax.scatter(measured_pareto['loaded_torque_mean'], measured_pareto['total_magnet_area_mm2'], s=70, label='Measured Pareto (11)')
ax.scatter(nsga_df['loaded_torque_mean'], nsga_df['total_magnet_area_mm2'], s=28, alpha=0.6, label='pymoo NSGA-II surrogate')
ax.set_xlabel('Average Torque [N.m]')
ax.set_ylabel('Total Magnet Area [mm^2]')
ax.set_title('pymoo NSGA-II: Torque vs Magnet Area')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

In [ ]:
nsga_df[['magnet_thickness_mm', 'magnet_length_mm', 'v_angle_deg', 'tip_gap_mm', 'loaded_torque_mean', 'loaded_torque_ripple_pct', 'cogging_torque_pkpk', 'total_magnet_area_mm2']].head(20)

In [ ]:
import plotly.express as px

surrogate_plot_df = pred_df.copy().reset_index(drop=True)
surrogate_plot_df['candidate_id'] = np.arange(1, len(surrogate_plot_df) + 1)
top_torque_surrogate_df = surrogate_plot_df.nlargest(12, 'loaded_torque_mean').copy()

print('Surrogate candidate rows =', len(surrogate_plot_df))
display(
    top_torque_surrogate_df[
        ['candidate_id', 'magnet_length_mm', 'v_angle_deg', 'magnet_thickness_mm', 'loaded_torque_mean', 'loaded_torque_ripple_pct', 'total_magnet_area_mm2']
    ]
)


In [ ]:
fig = px.scatter_3d(
    surrogate_plot_df,
    x='magnet_length_mm',
    y='magnet_thickness_mm',
    z='v_angle_deg',
    color='loaded_torque_mean',
    color_continuous_scale='Viridis',
    hover_data={
        'candidate_id': True,
        'magnet_length_mm': ':.3f',
        'v_angle_deg': ':.3f',
        'magnet_thickness_mm': ':.3f',
        'loaded_torque_mean': ':.3f',
        'loaded_torque_ripple_pct': ':.3f',
        'total_magnet_area_mm2': ':.3f',
    },
    title='Surrogate 1000-candidate 3D view: z = V angle, color = torque',
)
fig.update_traces(marker=dict(size=4.5, opacity=0.8))
fig.update_layout(
    scene=dict(
        xaxis_title='Length [mm]',
        yaxis_title='Thickness [mm]',
        zaxis_title='V angle [deg]',
    ),
    height=720,
)
fig.show()


In [ ]:
fig = px.scatter_3d(
    surrogate_plot_df,
    x='magnet_length_mm',
    y='magnet_thickness_mm',
    z='v_angle_deg',
    color='loaded_torque_ripple_pct',
    color_continuous_scale='Plasma_r',
    hover_data={
        'candidate_id': True,
        'magnet_length_mm': ':.3f',
        'v_angle_deg': ':.3f',
        'magnet_thickness_mm': ':.3f',
        'loaded_torque_mean': ':.3f',
        'loaded_torque_ripple_pct': ':.3f',
        'total_magnet_area_mm2': ':.3f',
    },
    title='Surrogate 1000-candidate 3D view: z = V angle, color = torque ripple',
)
fig.update_traces(marker=dict(size=4.5, opacity=0.8))
fig.update_layout(
    scene=dict(
        xaxis_title='Length [mm]',
        yaxis_title='Thickness [mm]',
        zaxis_title='V angle [deg]',
    ),
    height=720,
)
fig.show()


## Notes

Interpretation cautions:
- This notebook uses a surrogate trained on only 50 FEMM samples.
- Any 1000-candidate Pareto or `pymoo` NSGA Pareto here is **surrogate-based**, not direct FEMM truth.
- Use this notebook to pick promising regions, then validate those points with new FEMM runs.
